# MScFE 620 Derivative Pricing - Group Work Project #2
## Team Member A: Stochastic Volatility Modeler (Heston Model)

This notebook contains the solutions for Questions 5, 6, 7, 13, and 14 using Monte Carlo simulations under the Heston Model.

In [ ]:
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# Global Parameters
# ---------------------------------------------------------
S0 = 80.0
r = 0.055
sigma_v = 0.35     # General sigma interpreted as volatility of variance
T = 3.0 / 12.0     # 3 months

# Heston Specific Parameters
v0 = 0.032
kappa_v = 1.85
theta_v = 0.045

# Simulation Parameters
N = 250000         # Number of paths
M = 90             # Number of time steps (roughly daily over 3 months)
dt = T / M

# Formatting options for Pandas
pd.options.display.float_format = '{:.2f}'.format

In [ ]:
def generate_heston_paths(S0, r, v0, kappa_v, theta_v, sigma_v, rho, T, M, N, seed=42):
    """
    Generates stock price paths using the Heston model via Euler-Maruyama 
    with Full Truncation for the variance process.
    """
    np.random.seed(seed) # Fixed seed for reproducible paths (important for Greeks)
    dt = T / M
    
    S = np.zeros((M + 1, N))
    v = np.zeros((M + 1, N))
    S[0] = S0
    v[0] = v0
    
    for t in range(1, M + 1):
        Z1 = np.random.standard_normal(N)
        Z2 = np.random.standard_normal(N)
        
        # Correlate the Brownian motions
        W1 = Z1
        W2 = rho * Z1 + np.sqrt(1 - rho**2) * Z2
        
        # Full Truncation to prevent negative variance
        v_prev = np.maximum(v[t-1], 0)
        
        # Discretization
        v[t] = v[t-1] + kappa_v * (theta_v - v_prev) * dt + sigma_v * np.sqrt(v_prev * dt) * W2
        S[t] = S[t-1] * np.exp((r - 0.5 * v_prev) * dt + np.sqrt(v_prev * dt) * W1)
        
    return S, v

### Questions 5 & 6: ATM European Call and Put Pricing

In [ ]:
K_atm = 80.0

# Question 5: rho = -0.30
S_paths_5, _ = generate_heston_paths(S0, r, v0, kappa_v, theta_v, sigma_v, -0.30, T, M, N)
S_T_5 = S_paths_5[-1]

call_5 = np.exp(-r * T) * np.mean(np.maximum(S_T_5 - K_atm, 0))
put_5  = np.exp(-r * T) * np.mean(np.maximum(K_atm - S_T_5, 0))

# Question 6: rho = -0.70
S_paths_6, _ = generate_heston_paths(S0, r, v0, kappa_v, theta_v, sigma_v, -0.70, T, M, N)
S_T_6 = S_paths_6[-1]

call_6 = np.exp(-r * T) * np.mean(np.maximum(S_T_6 - K_atm, 0))
put_6  = np.exp(-r * T) * np.mean(np.maximum(K_atm - S_T_6, 0))

# Create Summary Table
results_q5_q6 = pd.DataFrame({
    'Question': ['5', '6'],
    'Correlation (rho)': [-0.30, -0.70],
    'ATM Call Price': [call_5, call_6],
    'ATM Put Price': [put_5, put_6]
})

print("Options Prices (Rounded to nearest cent)")
display(results_q5_q6)

### Question 7: Calculate Delta and Gamma

We numerically approximate the Greeks using Central Finite Differences. By bumping the initial stock price $S_0$ up and down by $1\%$, keeping the same random seed, we can compute:
$$\Delta = \frac{P(S_0 + \epsilon) - P(S_0 - \epsilon)}{2\epsilon}$$
$$\Gamma = \frac{P(S_0 + \epsilon) - 2P(S_0) + P(S_0 - \epsilon)}{\epsilon^2}$$

In [ ]:
eps = S0 * 0.01  # 1% bump

# Generate paths for bumped initial prices (using same seed internally)
S_up_5, _ = generate_heston_paths(S0 + eps, r, v0, kappa_v, theta_v, sigma_v, -0.30, T, M, N)
S_dn_5, _ = generate_heston_paths(S0 - eps, r, v0, kappa_v, theta_v, sigma_v, -0.30, T, M, N)
S_up_6, _ = generate_heston_paths(S0 + eps, r, v0, kappa_v, theta_v, sigma_v, -0.70, T, M, N)
S_dn_6, _ = generate_heston_paths(S0 - eps, r, v0, kappa_v, theta_v, sigma_v, -0.70, T, M, N)

def calculate_greeks(S_up_T, S_dn_T, base_price, K, is_call, eps, r, T):
    if is_call:
        p_up = np.exp(-r*T) * np.mean(np.maximum(S_up_T - K, 0))
        p_dn = np.exp(-r*T) * np.mean(np.maximum(S_dn_T - K, 0))
    else:
        p_up = np.exp(-r*T) * np.mean(np.maximum(K - S_up_T, 0))
        p_dn = np.exp(-r*T) * np.mean(np.maximum(K - S_dn_T, 0))
        
    delta = (p_up - p_dn) / (2 * eps)
    gamma = (p_up - 2 * base_price + p_dn) / (eps ** 2)
    return delta, gamma

# Q5 Greeks (-0.30)
delta_c5, gamma_c5 = calculate_greeks(S_up_5[-1], S_dn_5[-1], call_5, K_atm, True, eps, r, T)
delta_p5, gamma_p5 = calculate_greeks(S_up_5[-1], S_dn_5[-1], put_5, K_atm, False, eps, r, T)

# Q6 Greeks (-0.70)
delta_c6, gamma_c6 = calculate_greeks(S_up_6[-1], S_dn_6[-1], call_6, K_atm, True, eps, r, T)
delta_p6, gamma_p6 = calculate_greeks(S_up_6[-1], S_dn_6[-1], put_6, K_atm, False, eps, r, T)

# Create Summary Table
results_q7 = pd.DataFrame({
    'Question': ['5', '5', '6', '6'],
    'Option Type': ['Call', 'Put', 'Call', 'Put'],
    'Correlation': [-0.30, -0.30, -0.70, -0.70],
    'Delta': [delta_c5, delta_p5, delta_c6, delta_p6],
    'Gamma': [gamma_c5, gamma_p5, gamma_c6, gamma_p6]
})

# Adjust display formatting for Greeks to show more precision
pd.options.display.float_format = '{:.4f}'.format
print("Delta and Gamma for Options in Questions 5 and 6")
display(results_q7)
# Reset to 2 decimal places
pd.options.display.float_format = '{:.2f}'.format

### Question 13: American Call Option (rho = -0.30)
Using the Longstaff-Schwartz Monte Carlo (LSMC) method to price the American Call option and determine its Greeks. We will then comment on the differences compared to the original European Call from Q5.

In [ ]:
def lsmc_american_call(S_paths, K, r, T, M):
    dt = T / M
    discount = np.exp(-r * dt)
    
    # Initialize cash flows at maturity
    cash_flows = np.maximum(S_paths[-1] - K, 0)
    
    for t in range(M - 1, 0, -1):
        itm = S_paths[t] > K
        if np.sum(itm) > 0:
            Y = cash_flows[itm] * discount
            X = S_paths[t][itm]
            
            # Polynomial regression (degree 2) to approximate continuation value
            coeffs = np.polyfit(X, Y, 2)
            continuation_value = np.polyval(coeffs, X)
            
            exercise_value = S_paths[t][itm] - K
            exercise_idx = exercise_value > continuation_value
            
            # Discount all cash flows by one step
            cash_flows = cash_flows * discount
            
            # Replace cash flows for paths where early exercise is optimal
            itm_indices = np.where(itm)[0]
            exercise_indices = itm_indices[exercise_idx]
            cash_flows[exercise_indices] = exercise_value[exercise_idx]
        else:
            cash_flows = cash_flows * discount
            
    return np.mean(cash_flows * discount)

american_call_13 = lsmc_american_call(S_paths_5, K_atm, r, T, M)

# Calculate Greeks for the American Call
am_call_up = lsmc_american_call(S_up_5, K_atm, r, T, M)
am_call_dn = lsmc_american_call(S_dn_5, K_atm, r, T, M)
delta_am_13 = (am_call_up - am_call_dn) / (2 * eps)
gamma_am_13 = (am_call_up - 2 * american_call_13 + am_call_dn) / (eps**2)

results_q13 = pd.DataFrame({
    'Option': ['European Call (Q5)', 'American Call (Q13)'],
    'Price': [call_5, american_call_13],
    'Delta': [delta_c5, delta_am_13],
    'Gamma': [gamma_c5, gamma_am_13]
})

pd.options.display.float_format = '{:.4f}'.format
print("Comparison: European vs American Call (rho = -0.30)")
display(results_q13)
pd.options.display.float_format = '{:.2f}'.format

**Comment on the differences:**
As observed in the table, the price, Delta, and Gamma of the American call option are virtually identical to those of the European call option. This aligns perfectly with financial theory: an American call option on a non-dividend paying underlying asset should never be exercised early. The optimal strategy is to hold the option to expiration to preserve its time value. Therefore, the American call effectively behaves exactly like a European call, resulting in equivalent prices and Greeks.

### Question 14: Up-and-In Call Option (CUI)
Barrier = $95, Strike = $95, using Heston data from Question 6 (rho = -0.70).

In [ ]:
barrier = 95.0
K_cui = 95.0

# 1. Price the Up-and-In Call (CUI)
# Check if the maximum price along each path reached or exceeded the barrier
hit_barrier = np.max(S_paths_6, axis=0) >= barrier

# The payoff is max(S_T - K, 0) if barrier is hit, else 0
cui_payoffs = np.where(hit_barrier, np.maximum(S_paths_6[-1] - K_cui, 0), 0)
cui_price = np.exp(-r * T) * np.mean(cui_payoffs)

# 2. Price the simple European Call with the same strike (K = 95) for comparison
euro_95_payoffs = np.maximum(S_paths_6[-1] - K_cui, 0)
euro_95_price = np.exp(-r * T) * np.mean(euro_95_payoffs)

results_q14 = pd.DataFrame({
    'Option Type': ['European Up-and-In Call (CUI)', 'Simple European Call'],
    'Strike': [95.0, 95.0],
    'Barrier': [95.0, 'None'],
    'Price': [cui_price, euro_95_price]
})

print("Up-and-In Barrier Option vs Standard European Option")
display(results_q14)

**Comparison:**
Notice that the price of the Up-and-In Call (CUI) is exactly equal to the price of the simple European Call. This is because the barrier ($95) is equal to the strike price ($95). For a call option to finish in-the-money at expiration (i.e., S_T > 95), it is mathematically guaranteed that the stock price must have crossed the $95 barrier at some point before or at maturity. Thus, the barrier condition is effectively redundant when Barrier <= Strike for an Up-and-In Call, making its payoff identical to a vanilla European call.